# Creating Manifests and Archives

Two key features of the ProteinGym dataset are the manifest-based configuration and archiving of datasets. This notebook will demonstrate how to create manifest files, understand their structure, and create dataset archives. We'll use the NEIME 2019 dataset in `example_data/NEIME_2019` as our example. We'll cover how to create a manifest using your favorite text editor and how you can create the manifest through python. 

## What is a Manifest?

A manifest is a [TOML](https://toml.io/en/) configuration file that describes your dataset's structure and metadata. It serves as a blueprint for loading and organizing your protein data. TOML stands for Tom's Obvious Minimal Language and is designed as a configuration file that is easy to read and write. In this tutorial we will write the TOML file using the notebook cells, but you can also open the TOML in your favorite text editor.


## Manifest Structure

Let's examine the key sections of a manifest file:

In [ ]:
# First, let's look at the example manifest
from pathlib import Path

# Read the example manifest
manifest_path = Path("../example_data/neime_2019.toml")
manifest_content = manifest_path.read_text(encoding="utf-8")

print(manifest_content)

name = "NEIME_2019"
description = "The NEIME Kennouche 2019 (UniProt id: A0A1I9GEU1) datase"
version = "1.0.0"

[[ assay_conditions ]]
name = "PH"
description = "pH level of the samples"
unit = "pH"

[[ assay_conditions ]]
name = "T"
description = "Temperature level of the samples"
unit = "C"

[[ assays ]]
sequence = "mutated_sequence"
target = "DMS_score"
path = "./NEIME_2019/Assays/Assay1.csv"
[ assays.conditions ]
T = 37
PH = 7

[[ sequences ]]
path = "./NEIME_2019/sequences/A0A1I9GEU1.fasta"
type = "wild_type"
alphabet = "AA"

[[ structures ]]
path = "./NEIME_2019/Structures/computational.pdb"
name = "A0A1I9GEU1"
description = "ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN"

[structures.metadata]
source = "alphafold"
version = "v2.0"
avg_plddt = "90"
link = "alphafold.ebi.ac.uk/entry/A0A1I9GEU1"

[[ msas ]]
path = "./NEIME_2019/MSA/msa.a2m"
format = "fasta"


# Create a complete manifest using your favorite text editor

Let's create a complete manifest file step by step. Each blue block represents what you would write in your text editor.  

<div class="alert alert-block alert-warning">
<b>Usage of quotes:</b> If you are editing the toml file from a text editor, make sure you include quotes around all your named variables you assign, otherwise python will not recognize these as strings
</div>

A proper manifest should include:

### 1. Top-level Metadata

We give a version, name and description to the dataset. 

Here the version does not refer to the version of the dataset, but the version of the .toml schema. This is to keep track of the proteingym version that can read in the schemas. 

For the name we recommend the current ProteinGym format of `<Uniprot_ID>_<SPECIES>_<Author>_<Year>`. 

The description field we leave up to the author to add any information to the dataset that is not captured by the standard fields.


<div style="width: 70%">
  <div class="alert alert-block alert-info">
    version = "1.0.0" <br>
    name = "NEIME_2019" <br>
    description = "NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores." <br>
  </div>
</div>

### 2. Defining Assay Conditions

Here we define the assay_conditions that are used in our assays. In this example we add two entries, one for temperature and one for the pH. We can add as many sections as we want to indicate different assay conditions.
Each section starts with the `[[ assay_conditions ]]` indicator and contains the following entrys:

- **name**: Name of the condition.
- **description**: Description of the condition.
- **unit**: (SI) Unit of the condition - currently not enforced in anyway but in a future version it might be using [pint](https://pint.readthedocs.io/en/stable/).

Since our assay conditions can be different for each assay, we assign the value of each condition in the assay section.

<div style="width: 70%">
  <div class="alert alert-block alert-info">
[[ assay_conditions ]] <br>
name = "temperature" <br>
description = "Reaction temperature measured by Machine X in Lab Y" <br>
unit = "°C" <br>
[[ assay_conditions ]] <br>
name = "pH" <br>
description = "Buffer pH using Tris buffer" <br>
unit = "pH" <br>
  </div>
</div>


### 3. Assays and assigning conditions

An Assay contain relevant data from a single physical experiment - which protein variants were tested, in which conditions and the recorded statistics. At the dataset level, we link to a list of different assays.

Here we highlight the entry for the example DMS assay. Currently each dataset in ProteinGym consists of either a DMS or ClinVar assay, but for the case of protein engineering it can be beneficial to record multiple assays. **Support for multiple grouping multiple assays still in progress**

An Assay has three required fields:
- **sequence**: name of the column in your assay csv that contains the sequence.
- **target**: name of the column with quantity of interest (statistic).
- **path**: path to the location of your assay csv.

Note that the scope here is to keep track of data that is useful for the variant effect prediction challenge. It is not meant to serve as accurate description of what the assay or to make it reproducible.



Furthermore, we want to assign assay conditions to this assay. We can assign assay conditions in two methods, globally or per assay. In the global assignement we add a `value = number` to the `[[ assay_conditions ]]` section. This means all assays in the dataset follow these assay conditions.

We can also assign assay conditions per assay, for example in the case where you have performed assay1 at 30C and assay2 at 40C. We do this by adding a `[ assays.conditions ]` section to each assay:

**Global example**:
Here we assign a temperature of 37 to both Assay1 and Assay2:

<div style="width: 70%">
  <div class="alert alert-block alert-info">
[[ assay_conditions ]] <br>
name = "temperature" <br>
description = "Reaction temperature measured by Machine X in Lab Y" <br>
unit = "°C" <br>
value = 37

[[ assays ]] <br>
sequence = "mutated_sequence" <br>
target = "DMS_score" <br>
path = "./NEIME_2019/Assays/Assay1.csv" <br>

[[ assays ]] <br>
sequence = "mutated_sequence" <br>
target = "DMS_score" <br>
path = "./NEIME_2019/Assays/Assay2.csv" <br>

  </div>
</div>

**Per Assay Example**: Here we assign different temperatures to each Assay.

<div style="width: 70%">
  <div class="alert alert-block alert-info">
[[ assays ]] <br>
sequence = "mutated_sequence" <br>
target = "DMS_score" <br>
path = "./NEIME_2019/Assays/Assay1.csv" <br>

[ assays.conditions]<br>
T = 30<br>

[[ assays ]] <br>
sequence = "mutated_sequence" <br>
target = "DMS_score" <br>
path = "./NEIME_2019/Assays/Assay2.csv" <br>

[ assays.conditions]<br>
T = 40
  </div>
</div>

### 4. Sequences

We record the mutated sequences in sequences in the assay, but for some cases you might want to access the original sequence. For example, in the case of a constrastive learning mechanism you might want to contrast the mutated sequence to the wild-type sequence. For this you can add a sequence field to the dataset.

Each sequence section requires:
- **path**: path to the fasta or fastq file.
- **sequence_alphabet**: AA, DNA or RNA
- **sequence_type**: Indicate the type of the sequence, e.g. wild-type, engineered or a custom type

<div style="width: 70%">
  <div class="alert alert-block alert-info">
[[ sequences ]] <br>
path = "./NEIME_2019/sequences/A0A1I9GEU1.fasta" <br>
alphabet = "AA" <br>
type = "wild_type" <br>
  </div>
</div>

### 5. Structures

You can add structures to the dataset as either a directory or single files. We allow for the possibility of loading PDBs, Cifs, and binary Cifs. Since meta data on the structure is usually included in the structure file itself, we recommend to only include the meta data for datasets with single structures. If you are loading a directory of structures you can access the structure meta data through the structure object. We'll show you how to access the meta data in `03_Loading_and_Accessing_Data.ipynb`

The following four entries are allowed for structures:
- **path**: Path to the structure file or directory
- **name**: Name of the protein, e.g. PDB ID or Uniprot ID
- **description**: Description of the structure
- **metadata**: Metadata fields

You can add metadata as a seperate entry:

<div style="width: 70%">
  <div class="alert alert-block alert-info">
[[ structures ]] <br>
path = "./NEIME_2019/Structures/computational.pdb" <br>
name = "A0A1I9GEU1" <br>
description = "ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN" <br>

[ structures.metadata ] <br>
source = "alphafold" <br>
version = "v2.0" <br>
avg_plddt = "96" <br>
extra_fields = "extra_data" <br>
  </div>
</div>

<div class="alert alert-block alert-warning">
<b>Note:</b> If you are using a separate metadata block, be sure to use just single brackets.
</div>

### 6. Multiple Sequence Alignments (MSAs)

We allow for the possibility to add a single MSA to the dataset, representing the evolutionary alignments of the protein of interest. Here we are limited to file formats [supported](https://biopython.org/wiki/AlignIO) by BioPython.

The following entries are allowed for msas:
- **path (required)**: Path to the file
- **name**: The name of the MSA
- **description**: The description of the MSA.
- **format (required)**: The format of the MSA file
- **metadata**: The fields for metadata

<div style="width: 70%">
  <div class="alert alert-block alert-info">
[[ msas ]] <br>
path = "./NEIME_2019/MSA/msa.a2m" <br>
format = "fasta" <br>
name = "A0A1I9GEU1_NEIME" <br>
description = "Generated by Software X" <br>

[ msas.metadata ] <br>
software = "mmseqs2" <br>
sequence_identity = "30%" <br>
  </div>
</div>



## Writing the Manifest File

Now we put it all together and save it to our toml file.

In [2]:
complete_manifest = """version = "1.0.0"
name = "NEIME_2019"
description = "NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores"

[[ assay_conditions ]]
name = "T"
description = "Reaction temperature"
unit = "°C"

[[ assay_conditions ]]
name = "pH"
description = "Buffer pH"
unit = "pH"

[[ assays ]]
sequence = "mutated_sequence"
target = "DMS_score"
path = "./NEIME_2019/Assays/Assay1.csv"
[ assays.conditions ]
pH = 7
T = 37

[[ sequences ]]
path = "./NEIME_2019/sequences/A0A1I9GEU1.fasta"
alphabet = "AA"
type = "wild_type"

[[ structures ]]
path = "./NEIME_2019/Structures/computational.pdb"
name = "A0A1I9GEU1"
description = "ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN"

[ structures.metadata ]
source = "alphafold"
version = "v2.0"
avg_plddt = "96"
extra_fields = "extra_data"

[[ msas ]]
path = "./NEIME_2019/MSA/msa.a2m"
format = "fasta"
name = "A0A1I9GEU1_NEIME"
description = "Generated by Software X"

[ msas.metadata ]
software = "mmseqs2"
sequence_identity = "30%" """

In [ ]:
# Write the complete manifest
output_path = Path("../example_data/neime_2019_complete.toml")
output_path.write_text(complete_manifest, encoding="utf-8")

print(f"Complete manifest written to: {output_path}")

# Display the written content
print("\nGenerated manifest:")
print(output_path.read_text(encoding="utf-8"))

Complete manifest written to: ../example_data/neime_2019_complete.toml

Generated manifest:
version = "1.0.0"
name = "NEIME_2019"
description = "NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores"

[[ assay_conditions ]]
name = "T"
description = "Reaction temperature"
unit = "°C"

[[ assay_conditions ]]
name = "pH"
description = "Buffer pH"
unit = "pH"

[[ assays ]]
sequence = "mutated_sequence"
target = "DMS_score"
path = "./NEIME_2019/Assays/Assay1.csv"
[ assays.conditions ]
pH = 7
T = 37

[[ sequences ]]
path = "./NEIME_2019/sequences/A0A1I9GEU1.fasta"
alphabet = "AA"
type = "wild_type"

[[ structures ]]
path = "./NEIME_2019/Structures/computational.pdb"
name = "A0A1I9GEU1"
description = "ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN"

[ structures.metadata ]
source = "alphafold"
version = "v2.0"
avg_plddt = "96"
extra_fields = "extra_data"

[[ msas ]]
path = "./NEIME_2019/MSA/msa.a2m"
format = "fasta"
name = "A0A1I9GEU1_NEIME"
description = "Generated by Software X"

[ m

## Loading and Validating the Manifest

Let's load our manifest and validate it:

In [4]:
from pg2_dataset import Manifest, Dataset

# Load the manifest
manifest = Manifest.from_path(output_path)

print(f"Loaded manifest: {manifest.name}")
print(f"Description: {manifest.description}")
print(f"Version: {manifest.version}")
print(f"Number of assays: {len(manifest.assays)}")
print(f"Number of sequences: {len(manifest.sequences)}")
print(f"Number of structures: {len(manifest.structures)}")
print(f"Number of MSAs: {len(manifest.msas)}")
print(f"Number of assay conditions: {len(manifest.assay_conditions)}")

Loaded manifest: NEIME_2019
Description: NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores
Version: 1.0.0
Number of assays: 1
Number of sequences: 1
Number of structures: 1
Number of MSAs: 1
Number of assay conditions: 2


## Creating a Dataset from the Manifest

Now let's create a dataset from our manifest:

In [5]:
# Create dataset from manifest
try:
    dataset = Dataset.from_manifest(manifest)
    print(f"Successfully created dataset: {dataset.name}")
    print(f"Dataset contains:")
    print(f"  - {len(dataset.assays)} assays")
    print(f"  - {len(dataset.sequences)} sequences")
    print(f"  - {len(dataset.structures)} structures")
    print(f"  - {len(dataset.msas)} MSAs")
except Exception as e:
    print(f"Error creating dataset: {e}")
    print(
        "This might be due to missing data files. Check that all paths in the manifest exist."
    )

Successfully created dataset: NEIME_2019
Dataset contains:
  - 1 assays
  - 1 sequences
  - 1 structures
  - 1 MSAs


## Creating Dataset Archives

Once you have a working dataset, you can create a portable archive:

In [6]:
# Create an archive
archive_path = dataset.dump(path=Path("../example_data/"))
print(f"Dataset archived to: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / 1024:.1f} KB")

Dataset archived to: ../example_data/NEIME_2019.zip
Archive size: 1300.7 KB


## Archive Structure

Let's examine what's inside a dataset archive:

In [7]:
import zipfile

with zipfile.ZipFile(archive_path, "r") as zip_file:
    print("Archive contents:")
    for file_info in zip_file.filelist:
        print(f"  {file_info.filename} ({file_info.file_size} bytes)")

Archive contents:
  manifest.lock (874 bytes)
  assays/Assay1.csv (156447 bytes)
  sequences/tr|A0A1I9GEU1|A0A1I9GEU1_NEIME.fasta (245 bytes)
  structures/A0A1I9GEU1.pdb (97127 bytes)
  msas/A0A1I9GEU1_NEIME.fasta (1076529 bytes)


## Tips for Manifest Creation

### 1. File Path Organization
- Use relative paths in manifests for portability
- Organize data files in logical directories
- Keep manifest files at the root of your dataset directory

### 2. Naming Conventions
- Use descriptive names for datasets and assays
- Follow consistent naming patterns
- Include version information when appropriate

### 3. Metadata Completeness
- Always include descriptions for datasets and conditions
- Specify units for numerical conditions
- Document the source and processing of your data

### 4. Validation
- Always test your manifest by loading it
- Verify that all file paths exist and are accessible
- Check that assay conditions are properly defined



## Common Manifest Patterns

### Multiple Assays
```toml
[[assays]]
name = "binding_assay"
path = "assays/binding.csv"
target = "binding_affinity"

[[assays]]
name = "stability_assay"
path = "assays/stability.csv"
target = "melting_temp"
```

### Directory-based Data
```toml
[[structures]]
path = "structures/"  # All files in directory
```

### Complex Conditions
```toml
[[assay_conditions]]
name = "buffer_composition"
description = "Tris-HCl buffer with NaCl"
value = "50mM Tris-HCl, 150mM NaCl"
```




## Next Steps

Now that you know how to create manifests and archives, you can:

1. **Load and explore data**: See `03_Loading_and_Accessing_Data.ipynb`
2. **Create your own dataset**: Use your protein data with PG2 Dataset
3. **Share your work**: Distribute dataset archives to collaborators
4. **Use python to create your own dataset**: Keep on reading to see how we would use python to fully create the dataset

# Using python to create the full manifest

In [ ]:
from pg2_dataset import Manifest
from pg2_dataset.assay import AssayCondition, AssayManifestSection
from pg2_dataset.sequence import SequenceManifestSection
from pg2_dataset.structure import StructureManifestSection
from pg2_dataset.msa import MSAManifestSection, MSAFormat

### Add Assay Conditions

In [9]:
assay_conditions = [
    AssayCondition(name="T", description="Reaction temperature", unit="°C", value="37"),
    AssayCondition(name="pH", description="Buffer pH", unit="pH", value="7.4"),
]

### Add Assays

In [10]:
assays = [
    AssayManifestSection(
        sequence="mutated_sequence",
        target="DMS_score",
        path=Path("../example_data/NEIME_2019/Assays/Assay1.csv"),
    )
]

### Add reference sequences

Here we use constants to refer to the alphabet and sequence types.

In [11]:
sequences = [
    SequenceManifestSection(
        path=Path("../example_data/NEIME_2019/sequences/A0A1I9GEU1.fasta"),
        alphabet="AA",  # DNA, RNA, AA
        type="wild_type",  # wild_type, starting_sequence, engineered_sequence
    )
]

### Add Structures

In [12]:
structures = [
    StructureManifestSection(
        path=Path("../example_data/NEIME_2019/Structures/computational.pdb"),
        name="A0A1I9GEU1",
        description="ALPHAFOLD MONOMER V2.0 PREDICTION FOR PILIN",
        metadata={
            "source": "alphafold",
            "version": "v2.0",
            "avg_plddt": "96",
            "extra_fields": "extra_data",
        },
    )
]

### Add MSAs

In [13]:
msas = [
    MSAManifestSection(
        path=Path("../example_data/NEIME_2019/MSA/msa.a2m"),
        format=MSAFormat.FASTA,
        name="A0A1I9GEU1_NEIME",
        description="Generated by Software X",
        metadata={"software": "mmseqs2", "sequence_identity": "30%"},
    )
]

### Finally assemble it into the manifest:

In [14]:
manifest = Manifest(
    version="1.0.0",
    name="NEIME_2019",
    description="NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores",
    assay_conditions=assay_conditions,
    assays=assays,
    sequences=sequences,
    structures=structures,
    msas=msas,
)

In [15]:
# dump the manifest to file
from pathlib import Path

manifest.dump(path=Path("../example_data/neime_2019_python_example.toml"))

PosixPath('../example_data/neime_2019_python_example.toml')

### Load in the dataset from the created manifest

In [16]:
# Create dataset from manifest
try:
    dataset = Dataset.from_manifest(manifest)
    print(f"Successfully created dataset: {dataset.name}")
    print(f"Dataset contains:")
    print(f"  - {len(dataset.assays)} assays")
    print(f"  - {len(dataset.sequences)} sequences")
    print(f"  - {len(dataset.structures)} structures")
    print(f"  - {len(dataset.msas)} MSAs")
except Exception as e:
    print(f"Error creating dataset: {e}")
    print(
        "This might be due to missing data files. Check that all paths in the manifest exist."
    )

Successfully created dataset: NEIME_2019
Dataset contains:
  - 1 assays
  - 1 sequences
  - 1 structures
  - 1 MSAs
